In [ ]:
%pip install umap-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import umap.umap_ as umap
import os
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
%matplotlib inline

## Configuration

In [ ]:
dataset_name = "MNIST" # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
model_type = "Fine_Tuned" # Fine_Tuned | Base
layer_norm = "Pre_Post_Layer_Norm" # Pre_Post_Layer_Norm | Post_Layer_Norm
num_classes = 10

results_path = f"./Embedding_Captures/{dataset_name}/{domain}/{layer_norm}"
indices = [i for i in range(12)]

## File Prepping

In [ ]:
model_type = "Fine_Tuned"
fine_tuned_path = f"{results_path}/Entire_Transformation_Matrix_W/{model_type}"
results_fine_tuned = [] # File Loading

try:
    for filename in os.listdir(fine_tuned_path):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(fine_tuned_path, filename)
        if os.path.isfile(file_path):
            results_fine_tuned.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

results_fine_tuned = [pd.read_json(i) for i in results_fine_tuned]
results_fine_tuned = sorted(results_fine_tuned, key=lambda df: df["Label"][0])

In [ ]:
model_type = "Base"
base_path = f"{results_path}/Entire_Transformation_Matrix_W/{model_type}"
results_base = [] # File Loading

try:
    for filename in os.listdir(base_path):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(base_path, filename)
        if os.path.isfile(file_path):
            results_base.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

results_base = [pd.read_json(i) for i in results_base]
results_base = sorted(results_base, key=lambda df: df["Label"][0])

In [ ]:
fine_tuned_embeddings = {}
base_embeddings = {}
for i in range(num_classes):
        fine_tuned_embeddings[i] = np.array([np.array(embed) for embed in results_fine_tuned[i]["W"]])
        base_embeddings[i] = np.array([np.array(embed) for embed in results_base[0]["W"]])

In [ ]:
task_matrix = []
task_matrix_path = "./Class_Specific_Data/MNIST/Base_Fine_Tuned/Entire_Transformation_Matrix_W"

try:
    for filename in os.listdir(task_matrix_path):
        if filename in ["Standard_48000_Results_All_Classes.json"]:
            file_path = os.path.join(task_matrix_path, filename)
            if os.path.isfile(file_path):
                task_matrix.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{task_matrix_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

task_matrix = np.array([pd.read_json(i) for i in task_matrix][0]["W"][11])

## Augmentations

In [ ]:
# task_matrix (768, 768)
# fine_tuned_embeddings (10, 12, 768) 
# base_embeddings (10, 12, 768)

In [ ]:
augmented_embeddings = {}
for i in range(num_classes):
    augmented_embeddings[i] = []
    for j in indices:
        augmented_embeddings[i].append(np.array(base_embeddings[i][j] @ task_matrix))
    augmented_embeddings[i] = np.array(augmented_embeddings[i])

## Graphs

### Extraneous

In [ ]:
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(fine_tuned_embeddings[0])
aug_2d = pca.fit_transform(augmented_embeddings[0])
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color="blue")
plt.scatter(aug_2d[:,0], aug_2d[:, 1], color="green")

# Optionally label each point with its layer index
for i, (x, y) in enumerate(embeddings_2d):
    plt.text(x + 0.1, y, f"Layer {i}", fontsize=9)

for i, (x, y) in enumerate(aug_2d):
    plt.text(x + 0.1, y, f"Layer {i}", fontsize=9)

plt.title("PCA of 12 Layer Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()

In [ ]:
render = umap.UMAP(n_components=2)
vec_2d = render.fit_transform(fine_tuned_embeddings[0])
aug_2d = pca.fit_transform(augmented_embeddings[0])

plt.title("UMAP")
plt.scatter(vec_2d[:, 0], vec_2d[:, 1], color="blue")
plt.scatter(aug_2d[:,0], aug_2d[:, 1], color="green")

for i, (x, y) in enumerate(vec_2d):
    plt.text(x + 0.1, y, f"Layer {i}", fontsize=9)
for i, (x, y) in enumerate(aug_2d):
    plt.text(x + 0.1, y, f"Layer {i}", fontsize=9)
plt.show()

### Actual

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embeddings_2d = tsne.fit_transform(fine_tuned_embeddings[0])
t_2d = tsne.fit_transform(augmented_embeddings[0])

plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color="blue")
plt.scatter(t_2d[:, 0], t_2d[:, 1], color="green")
for i, (x, y) in enumerate(embeddings_2d):
    plt.text(x + 0.5, y, f"Layer {i}", fontsize=9)
for i, (x, y) in enumerate(t_2d):
    plt.text(x + 0.5, y, f"Layer {i}", fontsize=9)

plt.title("t-SNE of 12 Layer Embeddings")
plt.show()